# 07 — Análise de Dimensionalidade (PCA) (análise nova)

Verificação: os 10 indicadores de endurecimento representam uma dimensão única ou múltiplas?
Se 2+ componentes emergirem, o score composto pode estar escondendo estrutura.

**Fonte:** `data/processed/corpus_dataset.csv` (145 itens, 10 indicadores)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/processed/corpus_dataset.csv')

INDICATORS = ['desincorporacao', 'rigidez_postural', 'dessexualizacao',
              'uniformizacao_facial', 'heraldizacao', 'enquadramento_arquitetonico',
              'apagamento_narrativo', 'monocromatizacao', 'serialidade', 'inscricao_estatal']

INDICATOR_LABELS = {
    'desincorporacao': 'Desincorporação', 'rigidez_postural': 'Rigidez postural',
    'dessexualizacao': 'Dessexualização', 'uniformizacao_facial': 'Uniformização facial',
    'heraldizacao': 'Heraldicização', 'enquadramento_arquitetonico': 'Enquadramento arq.',
    'apagamento_narrativo': 'Apagamento narrativo', 'monocromatizacao': 'Monocromatização',
    'serialidade': 'Serialidade', 'inscricao_estatal': 'Inscrição estatal',
}

df_complete = df.dropna(subset=INDICATORS).copy()
X = df_complete[INDICATORS].values

# Standardize (PCA benefits from standardization even on same-scale data)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Itens: {len(df_complete)}")
print(f"Indicadores: {len(INDICATORS)}")
print(f"Escala original: 0–3 (todos) | StandardScaler aplicado para PCA")

## 7.1 Scree Plot — Variância Explicada

In [ ]:
pca = PCA()
pca.fit(X_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)
eigenvalues = pca.explained_variance_

# Kaiser criterion: eigenvalue > 1
n_kaiser = np.sum(eigenvalues > 1)
# 80% and 90% thresholds
n_80 = np.argmax(cumulative >= 0.80) + 1
n_90 = np.argmax(cumulative >= 0.90) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax = axes[0]
ax.bar(range(1, 11), explained, alpha=0.6, label='Variância individual')
ax.plot(range(1, 11), cumulative, 'o-', color='#C44E52', label='Variância acumulada')
ax.axhline(y=0.80, color='gray', linestyle='--', alpha=0.5, label='80%')
ax.axhline(y=0.90, color='gray', linestyle=':', alpha=0.5, label='90%')
ax.set_xlabel('Componente')
ax.set_ylabel('Variância explicada')
ax.set_title('Scree Plot — PCA dos 10 Indicadores')
ax.set_xticks(range(1, 11))
ax.legend()

# Eigenvalue plot
ax = axes[1]
ax.bar(range(1, 11), eigenvalues, alpha=0.6, color='#55A868')
ax.axhline(y=1, color='red', linestyle='--', label='Kaiser (λ=1)')
ax.set_xlabel('Componente')
ax.set_ylabel('Autovalor')
ax.set_title('Autovalores (Critério de Kaiser)')
ax.set_xticks(range(1, 11))
ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/fig_13_scree.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Componentes para 80% variância: {n_80}")
print(f"Componentes para 90% variância: {n_90}")
print(f"Componentes com autovalor > 1 (Kaiser): {n_kaiser}")
print(f"\nVariância por componente:")
for i in range(10):
    print(f"  PC{i+1}: {explained[i]*100:.1f}% (acumulada: {cumulative[i]*100:.1f}%, λ={eigenvalues[i]:.2f})")

## 7.2 Loadings — Quais indicadores carregam em cada componente?

In [ ]:
# Loadings matrix
loadings = pd.DataFrame(
    pca.components_.T,
    index=[INDICATOR_LABELS[c] for c in INDICATORS],
    columns=[f'PC{i+1}' for i in range(10)]
)

# Show first 4 components (enough to explain ~80%)
n_show = min(n_80 + 1, 4)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(loadings.iloc[:, :n_show], annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Loading'})
ax.set_title(f'Loadings dos Indicadores nos Primeiros {n_show} Componentes')
plt.tight_layout()
plt.savefig('../data/processed/fig_14_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

# Print loadings table
print("\nLoadings (|loading| > 0.30 são marcados):")
for comp in loadings.columns[:n_show]:
    print(f"\n  {comp} (var={explained[int(comp[2:])-1]*100:.1f}%):")
    for ind, val in loadings[comp].items():
        marker = '***' if abs(val) > 0.50 else '**' if abs(val) > 0.35 else ''
        print(f"    {ind:25s}: {val:+.3f} {marker}")

## 7.3 Biplot — Itens no Espaço PCA

In [ ]:
X_pca = pca.transform(X_scaled)
df_complete['PC1'] = X_pca[:, 0]
df_complete['PC2'] = X_pca[:, 1]

regime_colors = {'fundacional': '#4C72B0', 'normativo': '#DD8452',
                 'militar': '#C44E52', 'contra-alegoria': '#55A868'}

fig, ax = plt.subplots(figsize=(12, 8))

for regime, color in regime_colors.items():
    mask = df_complete['regime_iconocratico'] == regime
    ax.scatter(df_complete.loc[mask, 'PC1'], df_complete.loc[mask, 'PC2'],
               c=color, label=regime, alpha=0.7, s=50, edgecolors='white', linewidth=0.5)

# Loading vectors (scaled for visibility)
scale = 3
for i, ind in enumerate(INDICATORS):
    ax.annotate('', xy=(pca.components_[0, i]*scale, pca.components_[1, i]*scale),
                xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    ax.text(pca.components_[0, i]*scale*1.1, pca.components_[1, i]*scale*1.1,
            INDICATOR_LABELS[ind], fontsize=7, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
ax.set_title('Biplot PCA — Itens por Regime + Vetores de Loading')
ax.legend(title='Regime', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.3)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/fig_15_pca_biplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.4 Implicações para o Score Composto

### Interpretação:
- Se **1 componente** domina (>60% variância): o score composto de endurecimento é justificado como medida unidimensional
- Se **2+ componentes** emergem (>10% cada): o score composto pode estar escondendo subestrutura — considerar scores separados por dimensão
- A `monocromatização` no PC2 ou PC3 isolada confirmaria a descoberta da correlação fraca

### Decisão:
- [Preencher após execução: número de componentes, papel da monocromatização, recomendação para o score composto]